# JAE demo: denoising and manifold recovery on simulated data

This notebook uses **simulated** neural data, where the true low-dimensional
latent that generated the recording is known. That lets us check not just that
the model denoises, but that its learned latent actually recovers the underlying
manifold. On real recordings the true latent is unknown, so the recovery check
at the end is a **simulation-only validation**, not something you run on real data.

In [ ]:
import torch

from pyjae import JAE, factor_analysis_denoise, pca_denoise, simulate_neural_data
from pyjae.data import train_val_test_split
from pyjae.metrics import per_channel_vaf

# A 6D nonlinear manifold observed on 64 noisy channels. info['latents'] holds
# the true generative latent, which we only get because this is a simulation.
clean, noisy, info = simulate_neural_data(
    n_samples=400, n_channels=64, n_timepoints=96,
    latent_dim=6, snr_db=10.0, nonlinear=True, alpha=3.0, seed=0,
)
split = train_val_test_split(noisy, clean=clean, fracs=(0.7, 0.0, 0.3), seed=0)
train, test, test_clean = split['train']['noisy'], split['test']['noisy'], split['test']['clean']
print('data shape:', tuple(noisy.shape), '| true latent dim:', info['latent_dim'])

## Denoising: JAE1 vs the linear baselines

On nonlinear data the channel-split model should beat PCA and Factor Analysis.
On linear data it should not, since PCA is then near-optimal, so a win there
would be a red flag rather than a good sign.

In [ ]:
model = JAE(latent_dim=6, backend='jae1', verbose=False)
model.fit(train, epochs=200, batch_size=16)

jae1_vaf = per_channel_vaf(test_clean, model.denoise(test))['mean']
pca_vaf = per_channel_vaf(test_clean, pca_denoise(train, test, k=6))['mean']
fa_vaf = per_channel_vaf(test_clean, factor_analysis_denoise(train, test, k=6))['mean']
print(f'denoising VAF  ->  JAE1={jae1_vaf:.3f}   PCA={pca_vaf:.3f}   FA={fa_vaf:.3f}')

## Simulation-only validation: does the latent recover the true manifold?

JAE1 forces the two channel partitions to share a latent, so the averaged
(aligned) latent is the model's estimate of the underlying manifold. The true
generative latent is only identifiable up to a linear transform, so we score
recovery as the held-out R^2 of a linear map from the learned latent to the
true latent. This is only possible because we know the ground truth here.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression


def linear_recovery_r2(learned, true, train_frac=0.7):
    """Held-out R^2 of a linear map from a learned latent to the true latent."""
    k = int(train_frac * len(learned))
    pred = LinearRegression().fit(learned[:k], true[:k]).predict(learned[k:])
    t = true[k:]
    return 1 - ((t - pred) ** 2).sum() / ((t - t.mean(0)) ** 2).sum()


# True latent, flattened per timepoint to match the model's row ordering.
true_latent = torch.as_tensor(info['latents']).permute(0, 2, 1).reshape(-1, 6).numpy()

# JAE1 aligned latent on the full recording.
model.model.eval()
with torch.no_grad():
    out = model.model((noisy - model.mean_) / model.std_)
jae1_latent = ((out.latents[0] + out.latents[1]) / 2).numpy()

pca_scores = PCA(n_components=6).fit_transform(noisy.permute(0, 2, 1).reshape(-1, 64).numpy())

print(f'latent recovery R^2  ->  JAE1={linear_recovery_r2(jae1_latent, true_latent):.3f}   '
      f'PCA={linear_recovery_r2(pca_scores, true_latent):.3f}')